# Critical bosonic string integrand

This tutorial computes the critical bosonic string integrand in terms of the ribbon graph moduli:

$$
\left|\left\langle
\mathcal B_{\ell_1}\wedge\cdots\wedge
\mathcal B_{\ell_{6g-4}}c\widetilde c(0)
\right\rangle\right|
\left(Z_X^{(g)}\right)^{26}.
$$

It then compares the resulting expression at genus one and two to the known expression in terms of the period matrix.

In [1]:
import numpy as np

from string_amplitudes import (
    BCGhostCorrelatorData,
    RiemannSurfaceData,
    bc_correlator,
    bghost_measure,
    compute_period_map,
    critical_bosonic_string_integrand,
    generate_ribbon_graphs,
    genus_two_period_matrix_integrand,
    holomorphic_one_form_antiderivatives,
    matter_log_determinant,
    riemann_constant_vector,
)

# Genus 1

## Assemble holomorphic data

Construct the period matrix in terms of the ribbon graph parameters.

In [2]:
# one face means one puncture. At the moment, most of our algorithms
# only support one puncture.
graph = generate_ribbon_graphs(genus=1, n_faces=1)[0]
# edge_lengths is the ribbon graph edge lengths that are the moduli of the 
# Riemann surface.
edge_lengths = (20, 20, 20)
period_data = compute_period_map(
    graph,
    edge_lengths,
    period_quadrature_order=128,
)
# RiemannSurfaceData stores the holomorphic data in the disc frame.
# (all it is is a class that stores the data inputted below.)
surface = RiemannSurfaceData(
    genus=1,
    Omega=period_data.period_matrix,
    normalized_one_forms=period_data.normalized_one_forms,
    antiderivatives_normalized_forms=holomorphic_one_form_antiderivatives(
        period_data.normalized_one_forms,
        quadrature_order=96,
    ),
)
tau = complex(period_data.period_matrix[0, 0])

## Compute the $bc$ ghost measure

We now compute the bc ghost measure for the genus 1 string integrand. The raw value of the measure does not have intrinsic meaning until the full string integrand is computed.

In [3]:
# Construct the genus-one data used by the physical lambda=2 ghost correlator.
c_point = 0.0j
anchor_b_point = 0.12 + 0.08j
divisor_points = (0.25 + 0.10j,)
normalization_point = -0.20 + 0.15j
riemann_constant = riemann_constant_vector(surface)
theta_lattice_cutoff = 8
trial_ghost_data = BCGhostCorrelatorData(
    surface=surface,
    riemann_constant=riemann_constant,
    divisor_points=divisor_points,
    normalization_point=normalization_point,
    sigma_normalization=1.0,
    chiral_z1=1.0,
)
lambda_one_geometric_factor = bc_correlator(
    (anchor_b_point,),
    (c_point,),
    trial_ghost_data,
    lambda_weight=1.0,
    lattice_cutoff=theta_lattice_cutoff,
)
normalized_form_at_anchor = surface.normalized_one_forms[0](anchor_b_point)
chiral_z1 = (
    abs(lambda_one_geometric_factor / normalized_form_at_anchor) ** (2.0 / 3.0)
)
ghost_data = BCGhostCorrelatorData(
    surface=surface,
    riemann_constant=riemann_constant,
    divisor_points=divisor_points,
    normalization_point=normalization_point,
    sigma_normalization=1.0,
    chiral_z1=chiral_z1,
)
# num_integration_points is the number of integration points per segment
# used to compute \mathcal B.
num_integration_points = 32

# bghost_measure computes the bc ghost correlation function,
# with the positions of the b ghosts integrated over to form
# the component appearing in the string integrand.

# This coefficient has no standalone physical meaning before it is
# combined with the matter sector in the same conformal frame.
ghost_measure_value = bghost_measure(
    graph,
    edge_lengths,
    ghost_data,
    c_point=c_point,
    num_integration_points=num_integration_points,
    theta_lattice_cutoff=theta_lattice_cutoff,
)
print("bc ghost measure coefficient:", ghost_measure_value)

bc ghost measure coefficient: -47.59581900712467j


## Verify convergence of the bc ghost measure

Converting the b-ghost correlation function to the $\mathcal{B}$ object which includes the Beltrami differential requires integrating over the positions of the b ghosts along the seams in the disk frame. Here, we integrate over those positions.

In [4]:
# Four choices for the number of integration points.
integration_point_counts = (4, 8, 16, 32)
ghost_values = {
    count: bghost_measure(
        graph,
        edge_lengths,
        ghost_data,
        c_point=c_point,
        num_integration_points=count,
        theta_lattice_cutoff=theta_lattice_cutoff,
    )
    for count in integration_point_counts
}
reference = ghost_values[integration_point_counts[-1]]

# we can see that as the number of integration points, the integrated bc ghost
# correlation function converges to a definite value.

print("integration points    ghost measure magnitude    relative change")
for count in integration_point_counts:
    relative_change = abs(ghost_values[count] - reference) / abs(reference)
    print(
        f"{count:18d}    {abs(ghost_values[count]):23.10g}"
        f"    {relative_change:12.3e}"
    )

integration points    ghost measure magnitude    relative change
                 4                47.59566741       3.185e-06
                 8                47.59581901       1.747e-12
                16                47.59581901       5.837e-14
                32                47.59581901       0.000e+00


## Compute and verify the genus-one string integrand

With the genus one partition function given by

$$
Z_{X,\mathrm{disk}}^{(1)}
=
(2\pi)^{-1}
\frac{\tau_2^{-1/2}}{|Z_1|}.
$$

We compare the numerical result, in the canonical torus normalization, with the conventional expression

$$
(2\pi)^{-24}
\tau_2^{-13}
|\eta(\tau)|^{-48}
\left|
\frac{\partial(\tau,\overline{\tau})}
{\partial(\ell_1,\ell_2)}
\right|.
$$

In [5]:
# compute the dedekind eta function for the analytic comparison
def dedekind_eta(modulus, tolerance=1e-15):
    q = np.exp(2j * np.pi * modulus)
    q_abs = abs(q)
    factors = []
    q_power = q
    while abs(q_power) / (1.0 - q_abs) > tolerance:
        factors.append(1.0 - q_power)
        q_power *= q
    return np.exp(np.pi * 1j * modulus / 12.0) * np.prod(factors)

# compute the derivative of the modulus \tau with respect to
# \ell_i, i=1,2. Used to compute the Jacobian. Step size is the
# the finite difference in approximating the derivative.
def tau_derivatives_at_fixed_length(step):
    derivatives = []
    dependent_edge = len(edge_lengths) - 1
    for independent_edge in range(dependent_edge):
        plus = list(edge_lengths)
        minus = list(edge_lengths)
        plus[independent_edge] += step
        plus[dependent_edge] -= step
        minus[independent_edge] -= step
        minus[dependent_edge] += step
        tau_plus = compute_period_map(
            graph, plus, period_quadrature_order=128
        ).period_matrix[0, 0]
        tau_minus = compute_period_map(
            graph, minus, period_quadrature_order=128
        ).period_matrix[0, 0]
        derivatives.append((tau_plus - tau_minus) / (2.0 * step))
    return derivatives


derivatives_step_one = tau_derivatives_at_fixed_length(1)
derivatives_step_two = tau_derivatives_at_fixed_length(2)
d_tau_d_ell = [
    (4.0 * first - second) / 3.0
    for first, second in zip(derivatives_step_one, derivatives_step_two)
]
# the jacobian of \tau,\bar{\tau} with respect to \ell_1,\ell_2
tau_jacobian = (
    d_tau_d_ell[0] * np.conjugate(d_tau_d_ell[1])
    - d_tau_d_ell[1] * np.conjugate(d_tau_d_ell[0])
)


eta = dedekind_eta(tau)
matter_partition_per_scalar = (
    (2.0 * np.pi) ** (-1)
    * tau.imag ** (-0.5)
    / abs(chiral_z1)
)
genus_one_absolute_normalization = (2.0 * np.pi) ** 20
raw_disk_integrand = critical_bosonic_string_integrand(
    graph,
    edge_lengths,
    matter_partition_per_scalar,
    ghost_data,
    c_point=c_point,
    num_integration_points=num_integration_points,
    theta_lattice_cutoff=theta_lattice_cutoff,
)

# a factor that comes from the zero modes of the c-ghosts (special to genus one)
c_zero_mode_factor = abs(surface.normalized_one_forms[0](c_point)) ** 2
# assemble the entire numerical string integrand
numerical_integrand = (
    genus_one_absolute_normalization
    * c_zero_mode_factor
    * raw_disk_integrand
)
# assemble the entire analytic string integrand
analytic_integrand = (
    (2.0 * np.pi) ** (-24)
    * tau.imag ** (-13.0)
    * abs(eta) ** (-48)
    * abs(tau_jacobian)
)
relative_difference = abs(numerical_integrand - analytic_integrand) / analytic_integrand

print("tau:", tau)
print("numerical ribbon-graph integrand:", numerical_integrand)
print("analytic period-matrix integrand:", analytic_integrand)
print("relative difference:", relative_difference)
assert relative_difference < 1e-3

tau: (0.49999999999999994+0.8660254037844386j)
numerical ribbon-graph integrand: 1.5783413198715427e-17
analytic period-matrix integrand: 1.578270647248351e-17
relative difference: 4.4778519650518075e-05


# Genus 2

At genus two, the numerical expression is compared to the analytic expression in terms of the Igusa cusp form. We follow the conventions of Xi Yin's string notes, but exclude the factor of $\frac{1}{(8\pi)}$ stemming from the normalization of the sphere 2-point function.

## Assemble the data associated with a given ribbon graph

In [ ]:
# find one genus two ribbon graph
genus_two_graph = generate_ribbon_graphs(genus=2, n_faces=1)[0]
# here, we just do the computation for one set of edges.
genus_two_edge_lengths = (18, 20, 22, 24, 26, 28, 30, 32, 34)
genus_two_period_data = compute_period_map(
    genus_two_graph,
    genus_two_edge_lengths,
    period_quadrature_order=128,
)
# Assemble all the holomorphic data associated with the Riemann surface.
genus_two_surface = RiemannSurfaceData(
    genus=2,
    Omega=genus_two_period_data.period_matrix,
    normalized_one_forms=genus_two_period_data.normalized_one_forms,
    antiderivatives_normalized_forms=holomorphic_one_form_antiderivatives(
        genus_two_period_data.normalized_one_forms,
        quadrature_order=96,
    ),
)
# compute the matter partition function
matter_logdet = matter_log_determinant(
    genus_two_graph,
    genus_two_edge_lengths,
)
det_im_omega = np.linalg.det(genus_two_surface.Omega.imag)
# the chiral partition function as defined our paper
chiral_z1 = np.exp(0.5 * (matter_logdet - np.log(det_im_omega)))
matter_partition_per_scalar = (2 * np.pi) ** -2 * np.exp(-0.5 * matter_logdet)

# the coefficients of the powers z^n of the holomorphic one forms in the disc frame
coefficients = genus_two_surface.normalized_one_forms[0].coefficients
nonzero = np.flatnonzero(np.abs(coefficients) > 1e-10)
canonical_zeros = np.roots(coefficients[: nonzero[-1] + 1][::-1])
canonical_zeros = canonical_zeros[np.abs(canonical_zeros) < 0.99]
assert canonical_zeros.size == 2

# compute the riemann constant vector. "Filter divisors" is a set of points to select
# the Riemann constant vector.
riemann_constant = riemann_constant_vector(
    genus_two_surface,
    canonical_zeros,
    filter_divisors=((0.11 + 0.09j,),),
    lattice_cutoff=8,
)

# divisor_points is a set of arbitrary points to construct the
# sigma function in the VV formula (the final form is independent of the points)
divisor_points = (0.21 + 0.09j, -0.16 + 0.12j)
# normalization_point is a fixed reference point used to define the normalization
# of the sigma function
normalization_point = -0.19 + 0.13j
anchor_b_points = (0.12 + 0.08j, -0.11 + 0.17j)
anchor_c_point = normalization_point
# assembling all the data needed to compute the bc ghost correlator 
# (i.e. the inputs to the VV formula for \lambda=1)
trial_ghost_data = BCGhostCorrelatorData(
    surface=genus_two_surface,
    riemann_constant=riemann_constant,
    divisor_points=divisor_points,
    normalization_point=normalization_point,
    sigma_normalization=1.0,
    chiral_z1=chiral_z1,
)
# we compute the \lambda=1 VV correlator to fix the normalization of the VV \sigma function
# this is because the \lambda=1 correlator is independently equal to the determinant of the 
# determinant of the holomorphic one-forms.
trial_lambda_one = bc_correlator(
    anchor_b_points,
    (anchor_c_point,),
    trial_ghost_data,
    lambda_weight=1.0,
    lattice_cutoff=8,
)
one_form_matrix = np.asarray([
    [form(point) for point in anchor_b_points]
    for form in genus_two_surface.normalized_one_forms
])
sigma_normalization = (
    (2 * np.pi) ** 16
    * chiral_z1
    * np.linalg.det(one_form_matrix)
    / trial_lambda_one
)
# with the properly normalized sigma, 
# reassmble all the ghost data to compute the full correlation function
genus_two_ghost_data = BCGhostCorrelatorData(
    surface=genus_two_surface,
    riemann_constant=riemann_constant,
    divisor_points=divisor_points,
    normalization_point=normalization_point,
    sigma_normalization=sigma_normalization,
    chiral_z1=chiral_z1,
)

## Compare with the Igusa cusp form expression

We compare the numerical expression from the Verlinde-Verlinde formula,

$$
\left|
\left\langle
\mathcal B_{\ell_1}\wedge\cdots\wedge
\mathcal B_{\ell_8}\,
c\widetilde c(0)
\right\rangle
\right|
\left(Z_X^{(2)}\right)^{26},
$$

 to the known formula in terms of the Igusa cusp form:

$$
2^{28}(2\pi)^{-50}
\left|\det_{\mathbb R}\frac{\partial(\Omega_{11},\Omega_{12},\Omega_{22},u)}{\partial(\ell_1,\ldots,\ell_8)}\right|
\frac{1}{(\det\operatorname{Im}\Omega)^{13}|\widehat\omega_1(0)|^2|\chi_{10}(\Omega)|^2}.
$$

In [ ]:
# finally compute the full string integrand
disk_frame_density = critical_bosonic_string_integrand(
    genus_two_graph,
    genus_two_edge_lengths,
    matter_partition_per_scalar,
    genus_two_ghost_data,
    num_integration_points=3,
    theta_lattice_cutoff=8,
)
# then we compare to the known analytic formula in terms of the period matrix
period_matrix_density = genus_two_period_matrix_integrand(
    genus_two_graph,
    genus_two_edge_lengths,
    theta_lattice_cutoff=8,
)
# hopefully they match!
relative_difference = abs(disk_frame_density / period_matrix_density - 1)

print("Disk-frame density:   ", disk_frame_density)
print("Period-matrix density:", period_matrix_density)
print("Relative difference:  ", relative_difference)
assert relative_difference < 5e-2

Disk-frame density:    6.920744707672055e-47
Period-matrix density: 6.722304052875565e-47
Relative difference:   0.02951973805939412
